#### Ryanair Price Analysis: 
#### Scraping and Analyzing Ryanair Flight Prices

Merlin Mirley Moreno Palacios 
Date: 04/2026

This project analyzes flight price data collected from Ryanair through a web scraping process. The objective is to build an end-to-end data project that begins with data collection and continues through cleaning and exploratory analysis.


#### 1. Importing the required libraries

The scraping process begins by importing the Python libraries needed for browser automation, date handling, waiting times, and file storage.

Selenium is used to interact with Ryanair's website dynamically, allowing the script to load pages and extract flight information directly from the browser. Additional libraries such as `datetime` and `timedelta` are used to generate departure dates, while `time` is used to control temporary pauses during page loading.

Finally, the `csv` library is used to save the extracted observations in a structured tabular format for later cleaning and analysis.

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime, timedelta
import time
import csv

#### 2. Scraping configuration

Before collecting the data, I define the main parameters of the scraping process. These parameters determine the routes, time horizon, and output structure of the dataset.

First, I initialize the Selenium driver, which opens a Chrome browser that will be used to navigate the Ryanair website and extract flight information.

Next, I specify the list of destination airports to be analyzed, while keeping a fixed origin airport (Frankfurt-Hahn). I also define the time window of the analysis, starting from the day after the scraping date and covering a seven-day horizon.

Finally, I define the name of the output file and store the scraping date. Recording the scraping date is important because flight prices can vary over time, and this allows tracking when the data was collected.

In [ ]:
driver = webdriver.Chrome()

cities = ["ALC", "BCN"] # Add more city codes as needed
start_date = datetime.now() + timedelta(days=1) # Start from the day after today
num_days = 7 # Number of days to scrape data for
csv_filename = "flight_data.csv" # Output CSV file name, where the scraped data will be saved
ORIGIN_DEFAULT = "HHN" 
scrape_date = datetime.now().strftime("%Y-%m-%d")# scrape_date is the date when the data is being scraped, formatted as YYYY-MM-DD

#### 3. Scraping function: URL construction and page access

The core of the scraping process is implemented through a function that retrieves flight information for a given route and departure date.

The function takes three inputs: the origin airport, the destination airport, and the departure date. Using these parameters, it dynamically constructs the corresponding Ryanair URL. This is possible because the platform encodes search parameters such as origin, destination, and date directly in the URL.

Once the URL is created, Selenium is used to open the page in the browser. This allows the script to access the flight results that are displayed dynamically on the website.

In [ ]:
# Function to scrape flight data for a given origin, destination, and date
def scrape_flights(origin, destination, date_out): 
    url = (
        f"https://www.ryanair.com/es/es/trip/flights/select?"
        f"adults=1&teens=0&children=0&infants=0"
        f"&dateOut={date_out}&dateIn=&discount=0&isReturn=false"
        f"&promoCode=&isConnectedFlight=false"
        f"&originIata={origin}&destinationIata={destination}"
    )

    driver.get(url)

#### Handling page loading and dynamic content

Web pages do not load all their content instantly, especially in platforms like Ryanair that rely on dynamic elements. For this reason, it is necessary to introduce waiting mechanisms before attempting to extract any information.

I use Selenium's `WebDriverWait` to pause the execution until the page structure is loaded. Specifically, the script waits for the main body of the page to be present. In addition, I include a short manual delay to ensure that all flight information is fully rendered.

A try–except block is used to prevent the scraper from stopping in case the page does not load as expected. This makes the data collection process more robust.

In [ ]:
wait = WebDriverWait(driver, 25)

try:
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    time.sleep(3)
except:
    pass

#### Identifying flight result cards

Once the page has loaded, the next step is to identify the HTML elements that contain the flight results. These elements act as containers for the information associated with each available flight, such as departure time, arrival time, route, and price.

To do this, I use CSS selectors to search for the flight cards displayed on the Ryanair results page. Since website structures can vary slightly, the script includes a fallback selector in case the first one does not return any elements.

If no flight cards are found, the function returns an empty list and prints a message indicating that no flights were available for the selected route and date. This helps keep the scraping process stable and traceable.

In [ ]:
flight_cards = driver.find_elements(By.CSS_SELECTOR, "flight-card-new")
if not flight_cards:
    flight_cards = driver.find_elements(By.CSS_SELECTOR, ".flight-card-new")

if not flight_cards:
    print(f"No se encontraron vuelos para {origin} -> {destination} el {date_out}")
    return []

#### Extracting flight-level information

After identifying each flight card, the next step is to extract the relevant information contained within each element.

For each flight, I retrieve key variables such as departure time, arrival time, origin and destination cities, and the flight number. This is done by selecting specific HTML elements within each card using CSS selectors.

Special attention is given to price extraction. In some cases, flights may be sold out, and therefore no price is displayed. To handle this situation, the script checks for the presence of a "sold out" indicator and assigns a corresponding label instead of attempting to extract a price.

All extracted information is stored in a structured format and appended to a list, where each row represents a single flight observation. A try–except block ensures that the process continues even if some elements cannot be retrieved.

In [ ]:
flight_data = []

for card in flight_cards:
    try:
        departure_time = card.find_element(
            By.CSS_SELECTOR,
            '[data-ref="flight-segment.departure"] .flight-info__hour'
        ).text

        arrival_time = card.find_element(
            By.CSS_SELECTOR,
            '[data-ref="flight-segment.arrival"] .flight-info__hour'
        ).text

        origin_city = card.find_element(
            By.CSS_SELECTOR,
            '[data-ref="flight-segment.departure"] .flight-info__city'
        ).text

        destination_city = card.find_element(
            By.CSS_SELECTOR,
            '[data-ref="flight-segment.arrival"] .flight-info__city'
        ).text

        flight_number = card.find_element(
            By.CSS_SELECTOR, 
            ".card-flight-num__content"
        ).text

        sold_out = card.find_elements(By.CSS_SELECTOR, "flights-lazy-sold-out-flight-card")
        if sold_out:
            price = "sold out"
        else:
            price = card.find_element(
                By.CSS_SELECTOR, 
                ".flight-card-summary__new-value"
            ).text

        flight_data.append([
            scrape_date,
            flight_number,
            origin_city,
            origin,
            destination_city,
            destination,
            date_out,
            departure_time,
            arrival_time,
            price
        ])

    except Exception as e:
        print(f"Error extrayendo detalles ({origin}->{destination} {date_out}): {e}")
        return flight_data

#### 4. Running the scraper and storing the data

After defining the scraping function, the final step is to execute it across all route and date combinations and store the results in a structured CSV file.

The script first creates the output file and writes the column names that define the structure of the dataset. It then runs the scraper for two sets of routes: outbound flights from the default origin airport to the selected destinations, and inbound flights from those destinations back to the origin airport.

For each route, the script iterates over a seven-day departure window and calls the scraping function for each specific date. The extracted flight observations are then written row by row into the CSV file.

Finally, the browser is closed to ensure that the scraping session ends properly and system resources are released.

In [ ]:
try:
    with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow([
            "Scrape Date",
            "Flight Number",
            "Origin City",
            "Origin Airport",
            "Destination City",
            "Destination Airport",
            "Departure Date",
            "Departure Time",
            "Arrival Time",
            "Price"
        ])

        origin = ORIGIN_DEFAULT
        for destination in cities:
            for i in range(num_days):
                date_out = (start_date + timedelta(days=i)).strftime("%Y-%m-%d")
                flights = scrape_flights(origin, destination, date_out)
                for flight in flights:
                    writer.writerow(flight)

        destination = ORIGIN_DEFAULT
        for origin in cities:
            for i in range(num_days):
                date_out = (start_date + timedelta(days=i)).strftime("%Y-%m-%d")
                flights = scrape_flights(origin, destination, date_out)
                for flight in flights:
                    writer.writerow(flight)

    print(f"✅ Flight data has been saved to {csv_filename}.")

finally:
    driver.quit()